# Stat 248 Final Project Analysis

## Time-Series-Driven Predictive Maintenance with Reinforcement Learning

This notebook is the reproducible analysis companion for the final project codebase. It connects the project narrative to the underlying code and generated outputs.

The main question is: can time-series-based failure-risk estimation improve predictive maintenance decisions relative to reactive or fixed-schedule baselines?

## Reproducibility note

If the `outputs/` directory is already populated, the notebook can be used as a reporting notebook. If you want to regenerate everything from scratch, run the next code cell.

In [ ]:
from src.pdm_project.pipeline import run_project

# Uncomment the next line to regenerate all analysis outputs.
# run_project()

## Imports

In [ ]:
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd

## Project setup

The data are simulated because the repository originally contained only the proposal. The simulator creates degrading machines with a latent health process, a stressed regime, and three sensor streams: vibration, temperature, and pressure.

The analysis compares:

- `baseline_glm`: current age and raw sensors only
- `advanced_glm`: rolling summaries and Holt state features
- `nonlinear_ts_forest`: a nonlinear time-series model using richer temporal features

The decision layer then compares reactive maintenance, age-threshold replacement, a risk-threshold rule, and tabular Q-learning.

## Load output tables

In [ ]:
output_dir = Path('outputs')

risk_metrics = pd.read_csv(output_dir / 'risk_model_metrics.csv')
policy_summary = pd.read_csv(output_dir / 'policy_summary.csv')
stationarity = pd.read_csv(output_dir / 'stationarity_diagnostics.csv')
calibration = pd.read_csv(output_dir / 'calibration_summary.csv')
residual_acf = pd.read_csv(output_dir / 'residual_acf.csv')

## Predictive performance

This table evaluates short-horizon failure prediction. Better models should have higher AUC and lower Brier score / log loss.

In [ ]:
risk_metrics

The main time-series result is that richer temporal representation improves predictive quality. The nonlinear forest model performs best on the test set, which suggests the failure process contains nonlinear interactions among local trend, volatility, and recent history.

## Stationarity diagnostics

A first-difference transformation is a standard diagnostic step in time-series work. Here we check whether differenced sensor streams behave more like stationary series using the Augmented Dickey-Fuller test.

In [ ]:
stationarity

All three differenced sensor series reject the unit-root null strongly, which supports the use of local dynamic summaries such as rolling slopes, differences, and exponentially weighted averages.

## Calibration and residual diagnostics

A good predictive-maintenance model should produce probabilities that are not only discriminative, but also reasonably calibrated. Residual autocorrelation is also important because leftover time dependence can indicate a mis-specified model.

In [ ]:
calibration.head()

In [ ]:
residual_acf

## Example figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

for ax, name in zip(axes, ['example_trajectories.png', 'example_risk_path.png']):
    ax.imshow(mpimg.imread(output_dir / name))
    ax.axis('off')
    ax.set_title(name)

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

for ax, name in zip(axes, ['calibration_curve.png', 'policy_comparison.png']):
    ax.imshow(mpimg.imread(output_dir / name))
    ax.axis('off')
    ax.set_title(name)

plt.tight_layout()

## Maintenance policy comparison

The policy layer translates time-series forecasts into operational decisions. Lower average cost is better.

In [ ]:
policy_summary

The strongest result is that once the time-series risk signal becomes informative, a tuned risk-threshold policy performs extremely well. Q-learning is still competitive, but the project's central contribution is the time-series modeling layer rather than deep control.

## Conclusion

This project stays centered on time series: the best gains come from better temporal representation, better diagnostics, and better risk estimation. Reinforcement learning is included as a sequential decision layer, but it is not the main driver of performance.

For future work, a natural next step would be a real benchmark such as NASA C-MAPSS, or a more explicitly sequential statistical model such as a state-space model or hidden Markov model.